# **Temperature Annealing/Heating MD Directory Analysis Notebook**

This notebook provides a structured workflow for analyzing **Temperature Loop MD** simulations performed at different temperatures for a fixed number of particles and initial conditions, changing temperatures in a gradual process using thermostats. The analysis is carried out using modular Python scripts by reading the output and plotting variables which are produced by the [C++ NParticleMD program](directory/...).

---

## Workflow Overview
1. **Load and Extract Data**  
   Retrieve simulation output files based on the directory location.

2. **Visualize System Properties Over Time Steps/Temperatures**
   Plotting different variables to analyze system behavior during annealing and heating processes, including:
   - Energies (positional and rotational kinetic, potential, total)
   - Measured Temperature (positional and rotational degrees of freedom)
   - Average number of neighbors of the system for different shells and distances
   - Rotational Order Parameter ($\cos(2\Delta\Phi)$)
   - Linear and rotational velocity of the system's center of mass

3. **Visualize System Configuration**  
   Generate static and dynamic visualizations of particle configurations, allowing for direct comparison throughout the process by selecting target temperatures.

---

## Part 1: Import modules and choose directory
* Read key variables from the selected simulation directory.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import data_extraction
import visualization
# output_dir = "/home/hadis/selfassembly/run/particleOriented/july23_temploopGeo6/outputs"
output_dir = "/home/hadis/vector/output/Dec1_tempLoop_8patch/outputs"

In [ ]:
positions=data_extraction.read_variable(output_dir, 'positions')
kinetic_energy=data_extraction.read_variable(output_dir,'kinetic energy')
temperature=data_extraction.read_variable(output_dir, 'real temperature')
potential_energy=data_extraction.read_variable(output_dir,'potential energy')
orientation_order=data_extraction.read_variable(output_dir,'orientation order')
number_of_neighbors=data_extraction.read_variable(output_dir,'number of neighbors')
# com_velocity=data_extraction.read_variable(output_dir, 'center of mass velocity')
# com_ang_velocity=data_extraction.read_variable(output_dir, 'center of mass angular velocity')
handedness=data_extraction.read_variable(output_dir, 'handedness')

---

## Part 2: Visualize System Properties Over Time Steps/Temperatures
* Plot desired parameters to analyze system's trend over annealing and heating processes:

In [ ]:
visualization.plot_energies(kinetic_energy, potential_energy, temperature, show_potential=True, temperature_label=True)
visualization.plot_temperature(temperature)
visualization.plot_neighbors(number_of_neighbors, temperature, temperature_label=True)
visualization.plot_order_parameter(orientation_order, temperature, temperature_label=True)
# visualization.plot_com_velocity(com_velocity, temperature, temperature_label=True)
# visualization.plot_com_ang_velocity(com_ang_velocity, temperature, temperature_label=True)

---

## Part 3: Visualize System's Configuration
* Plot staics and dynamics configuration of system's particles based on the selected target temperature, and for the whole of process:


In [ ]:
def target_temperature_index (variable_name, target_temperature):
    data = variable_name['data']
    temperature_labels = variable_name['temperature label'].flatten()
    print(f"available temperatures are from {min(temperature_labels)} to {max(temperature_labels)}.")
          
    try:
        index = np.where(temperature_labels == target_temperature)[0][0]
    except IndexError:
        print(f"🞩🞩🞩 Target temperature {target_temperature} not found in temperature_labels. Please choose a valid temperature! 🞩🞩🞩")
        return 0

    total_snapshots = data.shape[0]
    return int(total_snapshots / len(temperature_labels) * index)

def plot_snapshot_temperature(positions, handedness, target_temperature, patchNums=None,
                               line_length=0.45, area=20, radius=True, color_palette='hsv', dpi=120):
    positions_data = positions['data']
    handedness_data = handedness['data']
    shot = target_temperature_index(positions, target_temperature)

    fig, ax = plt.subplots(figsize=(6, 4), dpi=dpi)
    
    x = positions_data[shot, :, 0]
    y = positions_data[shot, :, 1]
    phi = positions_data[shot, :, 2]  # orientation of each particle
    h = handedness_data[shot, :]

    # Define colors based on handedness
    colors = ['mistyrose' if val == 1 else 'lightsteelblue' for val in h]
    
    # Plot main particles
    ax.scatter(
        x, y, 
        s=110,
        edgecolors='black',
        facecolor=colors,
        alpha=0.8
    )

    if radius:
        if patchNums is not None:
            for a in range(patchNums):
                phi_a = phi + 2*np.pi * a / patchNums
                x_end = x + line_length * np.cos(phi_a)
                y_end = y + line_length * np.sin(phi_a)

                for i in range(len(x)):
                    ax.plot([x[i], x_end[i]], [y[i], y_end[i]], color='black', linewidth=0.8, alpha=0.8)
    # if radius:
    #     if patchNums is not None:
    #         phi = phi% (2*np.pi /patchNums)
    #         x_end = x + line_length * np.cos(phi)
    #         y_end = y + line_length * np.sin(phi)
    #         for i in range(len(x)):
    #             ax.plot([x[i], x_end[i]], [y[i], y_end[i]], color='black', linewidth=0.8, alpha=0.8)
                    
    else:
        # Fallback: color by orientation
        scatter = ax.scatter(
            x, y,
            c=phi,
            s=60,
            alpha=0.8,
            vmin=-np.pi,
            vmax=np.pi,
            cmap=color_palette
        )
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('φ in radian')

    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')
    ax.set_title(f'Configuration (T={target_temperature})')
    
    ax.set_xlim(0, area)
    ax.set_ylim(0, area)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_snapshot_temperature(positions, handedness, 0.08, patchNums=8, line_length=0.4, area=20, radius=True)

In [ ]:
visualization.plot_hist_phi_temperature(positions, 0.008, step_window=1000)

In [ ]:
visualization.plot_delta_phi_hist_temperature(positions, 0.008, min_dis=10, step_window=1000)

---
* Create **animation** of particle's movement over simulation evolution:

In [ ]:
visualization.animate_position_handedness(positions, handedness, 'tempLoop_8patch.mp4', line_length=0.38, area=20, color_p=False, frame_skip=20, frame_size=800, fps=20)